<a href="https://colab.research.google.com/github/yordanET/FUNDAMENTOS-DE-IA/blob/main/red_bayesiana_estudiante.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ejemplo de red bayesiana
Simulación para determinar si un estudiante aprobará o desaprobará.

HorasEstudio -> Asistencia -> NotaExamen -> Aprobacion

In [1]:
!pip install pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 12.0 MB/s eta 0:00:00


In [2]:
# libreria de redes bayesianas
# crear la red bayesiana
from pgmpy.models import DiscreteBayesianNetwork
# crear la tabla de probabilidades condicionales
from pgmpy.factors.discrete import TabularCPD
# realizar inferencia en la red bayesiana
from pgmpy.inference import VariableElimination

/usr/local/lib/python3.13/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


In [3]:
# crear la red bayesiana
model = DiscreteBayesianNetwork([
    ('HorasEstudio','Asistencia'),
    ('Asistencia','NotaExamen'),
    ('NotaExamen','Aprobacion')
])

In [4]:
# crear la tabla de probabilidades condicionales para la variable HorasEstudio
# 60% de probabilidad de estudiar pocas horas
# 40% de probabilidad de estudiar muchas horas
cpd_horas_estudio = TabularCPD(
    variable='HorasEstudio',
    variable_card=2,
    values=[[0.60],[0.40]],
    state_names={'HorasEstudio':['Pocas','Muchas']}
)

In [5]:
# crear la tabla de probabilidades condicionales para la variable Asistencia
# si estudia pocas horas: 80% asistencia baja y 20% asistencia alta
# si estudia muchas horas: 30% asistencia baja y 70% asistencia alta
cpd_asistencia = TabularCPD(
    variable='Asistencia',
    variable_card=2,
    values=[
        [0.80,0.30],
        [0.20,0.70]
    ],
    evidence=['HorasEstudio'],
    evidence_card=[2],
    state_names={
        'Asistencia':['Baja','Alta'],
        'HorasEstudio':['Pocas','Muchas']
    }
)

In [6]:
# crear la tabla de probabilidades condicionales para la variable NotaExamen
# con asistencia baja: 70% nota baja y 30% nota alta
# con asistencia alta: 20% nota baja y 80% nota alta
cpd_nota_examen = TabularCPD(
    variable='NotaExamen',
    variable_card=2,
    values=[
        [0.70,0.20],
        [0.30,0.80]
    ],
    evidence=['Asistencia'],
    evidence_card=[2],
    state_names={
        'NotaExamen':['Baja','Alta'],
        'Asistencia':['Baja','Alta']
    }
)

In [7]:
# crear la tabla de probabilidades condicionales para la variable Aprobacion
# con nota baja: 95% desaprueba y 5% aprueba
# con nota alta: 10% desaprueba y 90% aprueba
cpd_aprobacion = TabularCPD(
    variable='Aprobacion',
    variable_card=2,
    values=[
        [0.95,0.10],
        [0.05,0.90]
    ],
    evidence=['NotaExamen'],
    evidence_card=[2],
    state_names={
        'Aprobacion':['Desaprueba','Aprueba'],
        'NotaExamen':['Baja','Alta']
    }
)

In [8]:
# agregar al modelo las tablas de probabilidades condicionales
model.add_cpds(
    cpd_horas_estudio,
    cpd_asistencia,
    cpd_nota_examen,
    cpd_aprobacion
)

model.check_model()

True

In [9]:
print(model.get_cpds())
print('modelo de red bayesiana creado')

[<TabularCPD representing P(HorasEstudio:2) at 0x7856e1c82270>, <TabularCPD representing P(Asistencia:2 | HorasEstudio:2) at 0x7856917de350>, <TabularCPD representing P(NotaExamen:2 | Asistencia:2) at 0x7856917de490>, <TabularCPD representing P(Aprobacion:2 | NotaExamen:2) at 0x7856a4459e00>]
modelo de red bayesiana creado


In [10]:
# crear el motor de inferencia
inferencia = VariableElimination(model)

# calcular la probabilidad de aprobar si estudia muchas horas
# y tiene asistencia alta
resultado = inferencia.query(
    variables=['Aprobacion'],
    evidence={
        'HorasEstudio':'Muchas',
        'Asistencia':'Alta'
    }
)

print(resultado)

+------------------------+-------------------+
| Aprobacion             |   phi(Aprobacion) |
+========================+===================+
| Aprobacion(Desaprueba) |            0.2700 |
+------------------------+-------------------+
| Aprobacion(Aprueba)    |            0.7300 |
+------------------------+-------------------+


In [11]:
# consultar solamente la probabilidad de aprobar
probabilidad_aprobar = resultado.get_value(Aprobacion='Aprueba')

print('Probabilidad de aprobar:', probabilidad_aprobar)
print('Probabilidad de aprobar en porcentaje:', probabilidad_aprobar * 100, '%')

Probabilidad de aprobar: 0.7300000000000001
Probabilidad de aprobar en porcentaje: 73.00000000000001 %
